# Unified Table VI: one harness, 3-kind hard negatives, both datasets

This recomputes the entire SOTA comparison (Table VI) for CIFAR-10 and ImageNet under a single
methodology, so the whole table is internally consistent and matches TABLE 0 / Table I / Table V:

- **Hard negatives = noise + JPEG + blur (3 kinds)** on the clean test half (the same set used by the
  centerpiece and the reconciled Table I/V), replacing the earlier noise+JPEG-only Table VI.
- **3-feature aggregators** {z_HF, z_GL, z_PL} (SCAN removed): mean, median, pred-only (z_GL,z_PL),
  logistic* (supervised). Baselines: class-conditional Mahalanobis and LID.
- **Class-conditional Mahalanobis** with Ledoit-Wolf whitening. CIFAR-10 uses predicted labels (cache
  labels are degenerate; the classifier is accurate on clean CIFAR). ImageNet uses GT labels from the
  torchvision val set (set `IMAGENET_TV_ROOT`).
- Leakage-safe splits; paired bootstrap for the difference vs Mahalanobis and an above/tie/below verdict.

Why: the TOST run showed the previously reported CIFAR-10 "tie" (median 0.903) was inconsistent with
the 3-feature / current methodology (median ~0.85). This notebook produces the correct, consistent
numbers for both datasets so the manuscript can be updated honestly.



In [ ]:
# ===================== [PREAMBLE] =====================
import os, io as _io, subprocess, pickle, json
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision, torchvision.models as models, torchvision.transforms as T
from torchvision.models import ResNet50_Weights
from torchvision.transforms.functional import gaussian_blur
from PIL import Image as _Image
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.covariance import LedoitWolf
from scipy.spatial.distance import cdist

SEED=42; device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
np.random.seed(SEED); torch.manual_seed(SEED)

def _find(name, maxdepth=8):
    roots=['/home','/root','/workspace',os.path.expanduser('~'),'.','..','../..','.']
    res=[]
    for r in roots:
        if not os.path.exists(r): continue
        try:
            out=subprocess.run(['find',r,'-maxdepth',str(maxdepth),'-type','f','-name',name],
                               capture_output=True,text=True,timeout=20).stdout.strip()
            if out: res+=[p for p in out.split('\n') if p]
        except: pass
    return sorted(set(res))

CIFAR_MEAN=[0.4914,0.4822,0.4465]; CIFAR_STD=[0.2470,0.2435,0.2616]
IMGNET_MEAN=[0.485,0.456,0.406]; IMGNET_STD=[0.229,0.224,0.225]
def make_pp(ds):
    m,s=(CIFAR_MEAN,CIFAR_STD) if 'CIFAR' in ds else (IMGNET_MEAN,IMGNET_STD)
    mean=torch.tensor(m).view(1,3,1,1).to(device); std=torch.tensor(s).view(1,3,1,1).to(device)
    return lambda x:(x/255.0-mean)/std
def load_backbone(ds):
    cfg={'CIFAR-10':('resnet50_cifar10_finetuned.pt',10),'CIFAR-100':('resnet50_cifar100_finetuned.pt',100),
         'SVHN':('resnet50_svhn_finetuned.pt',10),'TinyImageNet':('resnet50_tinyimagenet_finetuned.pt',200)}
    if ds in cfg:
        ck=(_find(cfg[ds][0]) or [None])[0]; m=models.resnet50(weights=None); m.fc=nn.Linear(2048,cfg[ds][1])
        m.load_state_dict(torch.load(ck,map_location=device)['state_dict'])
    else:
        m=models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
    return m.to(device).eval()
def gb(x,s):
    k=int(2*np.ceil(3*s)+1); k=k+1 if k%2==0 else k
    return gaussian_blur(x,kernel_size=k,sigma=s)
def to224(img):
    if img.dim()==3: img=img.unsqueeze(0)
    img=img.float()
    if img.shape[-1]!=224: img=F.interpolate(img,size=(224,224),mode='bicubic',align_corners=False)
    return img.clamp(0,255)
def jpeg(x,q=75):
    a=x.detach().squeeze(0).permute(1,2,0).clamp(0,255).byte().cpu().numpy()
    b=_io.BytesIO(); _Image.fromarray(a).save(b,format='JPEG',quality=int(q)); b.seek(0)
    return torch.from_numpy(np.array(_Image.open(b).convert('RGB'))).float().permute(2,0,1)
def jpeg_batch(b,q=75): return torch.stack([jpeg(b[i:i+1]) for i in range(b.shape[0])]).to(b.device)
def median3(x):
    p=F.pad(x,(1,1,1,1),mode='reflect')
    return p.unfold(2,3,1).unfold(3,3,1).contiguous().view(*x.shape,9).median(-1).values

class Feats:
    def __init__(self, model):
        self.model=model; self.f={}
        self.h=[model.layer1.register_forward_hook(self._mk('l1')),
                model.layer2.register_forward_hook(self._mk('l2')),
                model.layer3.register_forward_hook(self._mk('l3')),
                model.layer4.register_forward_hook(self._mk('l4'))]
    def _mk(self,n):
        def hook(m,i,o): self.f[n]=o.mean((2,3)).detach()
        return hook
    def __call__(self,x):
        self.f={}
        with torch.no_grad(): self.model(x)
        return [self.f['l1'],self.f['l2'],self.f['l3'],self.f['l4']]

def extract(imgs, fe, bb, pp, glsig, bs=64):
    L=[[],[],[],[]]; H=[];G=[];P=[]
    for i in range(0,len(imgs),bs):
        b=torch.cat(imgs[i:i+bs],0).to(device)
        with torch.no_grad():
            fl=fe(pp(b))
            for j in range(4): L[j].append(fl[j].cpu().numpy())
            p0=F.softmax(bb(pp(b)),1); p1=F.softmax(bb(pp(gb(b,glsig))),1)
            G.append((p0-p1).abs().sum(1).cpu().numpy())
            sq=median3(jpeg_batch(b)).clamp(0,255); p2=F.softmax(bb(pp(sq)),1)
            P.append((p0-p2).abs().sum(1).cpu().numpy())
            H.append(((b-gb(b,0.5)).abs().flatten(1).mean(1)/255.0).cpu().numpy())
    return [np.concatenate(x) for x in L], np.concatenate(H),np.concatenate(G),np.concatenate(P)

def auc_ci(neg,pos,B=2000,seed=SEED):
    neg=np.asarray(neg);pos=np.asarray(pos)
    y=np.r_[np.zeros(len(neg)),np.ones(len(pos))]; base=roc_auc_score(y,np.r_[neg,pos])
    rng=np.random.RandomState(seed); b=[]
    for _ in range(B):
        ni=rng.randint(0,len(neg),len(neg)); pi=rng.randint(0,len(pos),len(pos))
        b.append(roc_auc_score(y,np.r_[neg[ni],pos[pi]]))
    return base,float(np.percentile(b,2.5)),float(np.percentile(b,97.5))
def paired_diff_ci(negA,posA,negM,posM,B=2000,seed=SEED):
    negA,posA,negM,posM=map(np.asarray,(negA,posA,negM,posM))
    y=np.r_[np.zeros(len(negA)),np.ones(len(posA))]
    base=roc_auc_score(y,np.r_[negA,posA])-roc_auc_score(y,np.r_[negM,posM])
    rng=np.random.RandomState(seed); d=[]
    for _ in range(B):
        ni=rng.randint(0,len(negA),len(negA)); pi=rng.randint(0,len(posA),len(posA))
        yy=np.r_[np.zeros(len(ni)),np.ones(len(pi))]
        a=roc_auc_score(yy,np.r_[negA[ni],posA[pi]]); m=roc_auc_score(yy,np.r_[negM[ni],posM[pi]])
        d.append(a-m)
    return base,float(np.percentile(d,2.5)),float(np.percentile(d,97.5))
def half(n,seed=SEED):
    rng=np.random.RandomState(seed); idx=np.arange(n); rng.shuffle(idx); return idx[:n//2], idx[n//2:]
def find_mixed():
    out={}
    for p in _find('mixed_dataset.pkl'):
        pl=p.lower()
        if ('cifar10' in pl or 'cifar_10' in pl): out.setdefault('CIFAR-10',p)
        elif 'imagenet' in pl and 'eps8' in pl: out.setdefault('ImageNet',p)
    return out
print('[PREAMBLE] ready')


In [ ]:
# ===================== class-conditional Mahalanobis (Ledoit-Wolf whitening) =====================
def fit_maha_cc(Fl, labels):
    labels=np.asarray(labels); cls=np.unique(labels)
    assert len(cls)>1, 'calibration has <=1 class (degenerate labels)'
    Ws=[]; WMs=[]; coverage={int(c):int((labels==c).sum()) for c in cls}
    for f in Fl:
        mu=np.stack([f[labels==c].mean(0) for c in cls])
        cen=np.concatenate([f[labels==c]-mu[i] for i,c in enumerate(cls)],0)
        cov=LedoitWolf().fit(cen).covariance_; L=np.linalg.cholesky(np.linalg.inv(cov))
        Ws.append(L); WMs.append(mu.dot(L))
    return Ws, WMs, cls, coverage
def maha(Fl, Ws, WMs):
    s=0
    for f,L,WM in zip(Fl,Ws,WMs):
        w=f.dot(L); d2=(w**2).sum(1)[:,None]+(WM**2).sum(1)[None,:]-2.0*w.dot(WM.T)
        s=s+(-d2.min(1))
    return s
print('[maha] ready')


In [ ]:
# ===================== [CONFIG] + calibration builders =====================
CONFIG = {
    "IMAGENET_TV_ROOT": "./data/imagenet/",   # SET ME: folder with meta.bin + ILSVRC2012_img_val.tar (GT labels)
    "N_PER_CLASS": 20,
    "MAX_CLASSES": 1000,
    "CIFAR_CALIB_N": 2000,      # clean CIFAR images for predicted-label Mahalanobis calibration
}
def build_calib_from_torchvision(root, n_per_class, max_classes):
    tf=T.Compose([T.Resize(256),T.CenterCrop(224),T.ToTensor()])
    base=torchvision.datasets.ImageNet(root, split='val', transform=tf)
    by_cls={}
    for i,lab in enumerate(base.targets): by_cls.setdefault(int(lab),[]).append(i)
    cls=sorted(by_cls.keys())[:max_classes]
    rng=np.random.RandomState(SEED); imgs=[]; labs=[]
    for c in cls:
        ids=by_cls[c]; rng.shuffle(ids)
        for i in ids[:n_per_class]:
            x,_=base[i]; imgs.append((x*255.0).unsqueeze(0)); labs.append(c)
    return imgs, np.array(labs)
def build_calib_cifar_pred(mixed, bb, pp, n):
    imgs=[to224(im).cpu() for (im,lb,atk) in mixed if atk=='clean']
    rng=np.random.RandomState(SEED); idx=rng.permutation(len(imgs))[:n]; imgs=[imgs[i] for i in idx]
    labs=[]
    for i in range(0,len(imgs),64):
        b=torch.cat(imgs[i:i+64],0).to(device)
        with torch.no_grad(): labs.append(bb(pp(b)).argmax(1).cpu().numpy())
    return imgs, np.concatenate(labs)
print('[config] ready  | IMAGENET_TV_ROOT =', CONFIG["IMAGENET_TV_ROOT"])


In [ ]:
# ===================== run unified Table VI for both datasets (3-kind hard negatives) =====================
MX=find_mixed()
DSETS=[('CIFAR-10',0.5),('ImageNet',1.0)]
OUT='./sota_unified'; os.makedirs(OUT,exist_ok=True)
RESULTS={}

def lid_factory(ref_layer4):
    def lid_batch(query):
        D=np.sort(cdist(query,ref_layer4),axis=1)[:,:21]; r=D/(D[:,-1:]+1e-12)
        return -1.0/(np.mean(np.log(r[:,:-1]+1e-12),axis=1)+1e-12)
    return lid_batch

for ds,glsig in DSETS:
    if ds not in MX: print('skip',ds,'(no mixed_dataset)'); continue
    print('\n==============', ds, '==============')
    bb=load_backbone(ds); pp=make_pp(ds); fe=Feats(bb)
    mixed=pickle.load(open(MX[ds],'rb'))
    clean=[(to224(im).cpu(),lb) for (im,lb,atk) in mixed if atk=='clean']
    advs =[to224(im).cpu() for (im,lb,atk) in mixed if atk!='clean']
    rng=np.random.RandomState(SEED); clean=[clean[i] for i in rng.permutation(len(clean))[:500]]
    ci,ti=half(len(clean)); te=[clean[i] for i in ti]; te_imgs=[x[0] for x in te]
    # ---- 3-kind hard negatives on the clean test half ----
    noise=[(x+torch.randn_like(x)*8.0).clamp(0,255) for x in te_imgs]
    jp   =[jpeg(x.to(device)).cpu().unsqueeze(0) for x in te_imgs]
    bl   =[gb(x.to(device),1.0).clamp(0,255).cpu() for x in te_imgs]
    hn=noise+jp+bl
    # ---- Mahalanobis calibration ----
    if ds=='ImageNet':
        assert CONFIG["IMAGENET_TV_ROOT"], 'set CONFIG["IMAGENET_TV_ROOT"] for ImageNet GT labels'
        calib_imgs,calib_labs=build_calib_from_torchvision(CONFIG["IMAGENET_TV_ROOT"],CONFIG["N_PER_CLASS"],CONFIG["MAX_CLASSES"])
    else:
        calib_imgs,calib_labs=build_calib_cifar_pred(mixed, bb, pp, CONFIG["CIFAR_CALIB_N"])
    # ---- features ----
    Fcal,_,_,_   = extract(calib_imgs, fe, bb, pp, glsig)
    Ft,Ht,Gt,Pt  = extract(te_imgs,    fe, bb, pp, glsig)
    Fa,Ha,Ga,Pa  = extract(advs,       fe, bb, pp, glsig)
    Fh,Hh,Gh,Ph  = extract(hn,         fe, bb, pp, glsig)
    Fc2,Hc,Gc,Pc = extract([x[0] for x in [clean[i] for i in ci]], fe, bb, pp, glsig)
    mu={'h':Hc.mean(),'g':Gc.mean(),'p':Pc.mean()}; sd={'h':Hc.std()+1e-8,'g':Gc.std()+1e-8,'p':Pc.std()+1e-8}
    def Z(H,G,P): return np.stack([(H-mu['h'])/sd['h'],(G-mu['g'])/sd['g'],(P-mu['p'])/sd['p']],1)
    Zt,Za,Zh,Zc = Z(Ht,Gt,Pt),Z(Ha,Ga,Pa),Z(Hh,Gh,Ph),Z(Hc,Gc,Pc)
    # ---- Mahalanobis ----
    Ws,WMs,cls,coverage=fit_maha_cc(Fcal,calib_labs); covv=np.array(list(coverage.values()))
    print(f'  Maha calib: {len(cls)} classes, per-class min/median/max={covv.min()}/{int(np.median(covv))}/{covv.max()}')
    # ---- leakage-safe splits + logistic ----
    ai_c,ai_t=half(len(advs),seed=1); hi_c,hi_t=half(len(hn),seed=2)
    Xtr=np.concatenate([Zc,Zh[hi_c],Za[ai_c]],0); ytr=np.r_[np.zeros(len(Zc)+len(hi_c)),np.ones(len(ai_c))]
    lr=LogisticRegression(max_iter=2000,class_weight='balanced').fit(Xtr,ytr)
    lid=lid_factory(Fc2[3])
    def scores(which):
        if which=='Mahalanobis':
            st=-maha(Ft,Ws,WMs); sa=-maha([f[ai_t] for f in Fa],Ws,WMs); sh=-maha([f[hi_t] for f in Fh],Ws,WMs)
        elif which=='LID':
            st=lid(Ft[3]); sa=lid(Fa[3][ai_t]); sh=lid(Fh[3][hi_t])
        elif which=='mean': st,sa,sh=Zt.mean(1),Za[ai_t].mean(1),Zh[hi_t].mean(1)
        elif which=='median': st,sa,sh=np.median(Zt,1),np.median(Za[ai_t],1),np.median(Zh[hi_t],1)
        elif which=='pred-only': st,sa,sh=Zt[:,[1,2]].mean(1),Za[ai_t][:,[1,2]].mean(1),Zh[hi_t][:,[1,2]].mean(1)
        elif which=='logistic':
            f=lambda Zx: lr.predict_proba(Zx)[:,1]; st,sa,sh=f(Zt),f(Za[ai_t]),f(Zh[hi_t])
        return np.concatenate([st,sh]), sa
    negM,posM=scores('Mahalanobis'); TABLE={}
    for which in ['Mahalanobis','LID','mean','median','pred-only','logistic']:
        neg,pos=scores(which); a,lo,hi=auc_ci(neg,pos)
        if which=='Mahalanobis': diff='(reference)'; verdict='-'
        else:
            d,dlo,dhi=paired_diff_ci(neg,pos,negM,posM)
            diff=f'{d:+.3f} [{dlo:+.3f}, {dhi:+.3f}]'; verdict='above' if dlo>0 else ('tie' if (dlo<=0<=dhi) else 'below')
        TABLE[which]={'auroc':round(float(a),4),'ci':[round(float(lo),4),round(float(hi),4)],'diff_vs_maha':diff,'verdict':verdict}
        print(f'    {which:<12} {a:.3f} [{lo:.3f},{hi:.3f}]   diff {diff}   {verdict}')
    print('    logistic weights [z_HF,z_GL,z_PL]:', np.round(lr.coef_[0],3).tolist())
    RESULTS[ds]={'table':TABLE,'logistic_weights':lr.coef_[0].tolist(),
                 'maha_coverage':{'n_classes':int(len(cls)),'per_class_min':int(covv.min()),
                                  'per_class_median':int(np.median(covv)),'per_class_max':int(covv.max())},
                 'hard_negatives':'noise+jpeg+blur'}

json.dump(RESULTS, open(os.path.join(OUT,'table6_unified.json'),'w'), indent=2)
print('\nsaved', os.path.join(OUT,'table6_unified.json'))
